Date: 01/02/2024 <br>
Desc: To train BN model for UAV communication reliability prediction, using pandas crosstab to populate CPTs.

## Training Scripts

In [1]:
import pandas as pd
import numpy as np 
import math
import os
from tqdm import tqdm
from sklearn.naive_bayes import MultinomialNB
from joblib import dump
from multiprocessing.pool import Pool
from itertools import repeat

def generate_reliability_dataset(dataset_details_df):
    '''
    Duplicate each row in dataset_details_df based on the recorded number of samples
    Returns a dataset DF with the inputs and outputs needed to train the BN
    '''
    df_train_list = []
    for row in tqdm(dataset_details_df.itertuples()):
        hdist = row.Horizontal_Distance_Class
        height = row.Height_Class
        uav_send_int = row.UAV_Sending_Interval_Class
        mcs = row.MCS
        num_reliable = row.Num_Reliable
        num_delay_excd = row.Num_Delay_Excd
        num_incr_rcvd = row.Num_Incr_Rcvd
        num_q_overflow = row.Num_Q_Overflow

        if num_reliable > 0:
            reliable_packets = pd.DataFrame({"Horizontal_Distance_Class": hdist, "Height_Class": height, "UAV_Sending_Interval_Class": uav_send_int, "MCS": mcs, "Packet_State": 0}, index=[0])
            reliable_packets = reliable_packets.loc[reliable_packets.index.repeat(num_reliable)]
        else:
            reliable_packets = pd.DataFrame({})

        if num_delay_excd > 0:
            delay_excd_packets = pd.DataFrame({"Horizontal_Distance_Class": hdist, "Height_Class": height, "UAV_Sending_Interval_Class": uav_send_int, "MCS": mcs, "Packet_State": 1}, index=[0])
            delay_excd_packets = delay_excd_packets.loc[delay_excd_packets.index.repeat(num_delay_excd)]
        else:
            delay_excd_packets = pd.DataFrame({})

        if num_q_overflow > 0:
            q_overflow_packets = pd.DataFrame({"Horizontal_Distance_Class": hdist, "Height_Class": height, "UAV_Sending_Interval_Class": uav_send_int, "MCS": mcs, "Packet_State": 2}, index=[0])
            q_overflow_packets = q_overflow_packets.loc[q_overflow_packets.index.repeat(num_q_overflow)]
        else:
            q_overflow_packets = pd.DataFrame({})

        if num_incr_rcvd > 0:
            incr_rcvd_packets = pd.DataFrame({"Horizontal_Distance_Class": hdist, "Height_Class": height, "UAV_Sending_Interval_Class": uav_send_int, "MCS": mcs, "Packet_State": 3}, index=[0])
            incr_rcvd_packets = incr_rcvd_packets.loc[incr_rcvd_packets.index.repeat(num_incr_rcvd)]
        else:
            incr_rcvd_packets = pd.DataFrame({})
        df_train_list.append(pd.concat([reliable_packets, delay_excd_packets, q_overflow_packets, incr_rcvd_packets]))

    df_train = pd.concat(df_train_list)
    return df_train

def get_mcs_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["MCS"] = ''
    df.loc[(df["Modulation"] == "BPSK") & (df["Bitrate"] == 6.5), "MCS"] = 0 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 13), "MCS"] = 1 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 19.5), "MCS"] = 2 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 26), "MCS"] = 3 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 39), "MCS"] = 4 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 52), "MCS"] = 5 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 58.5), "MCS"] = 6 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 65), "MCS"] = 7 # MCS Index 0

    return df

def process_dataset(dataset_path, hdist_num_bins=121, height_num_bins=9):
    '''
    Process dataset for training BN Model
    Quantize input data to classes
    Generate dataset from dataset_details_df
    '''
    # Read input and output data from dataset details
    df_dtypes = {"Horizontal_Distance": np.float64, "Height": np.int16,	"U2G_Distance": np.int32, "UAV_Sending_Interval": np.float64, "Mean_SINR": np.float64, "Std_Dev_SINR": np.float64,
                "Num_Sent": np.int32, "Num_Reliable": np.int32, "Num_Delay_Excd": np.int32, "Num_Incr_Rcvd": np.int32, "Num_Q_Overflow": np.int32, "Modulation": str, "Bitrate": np.float64}
    dataset_details_df = pd.read_csv(dataset_path, 
                                usecols = ["Horizontal_Distance", "Height", "UAV_Sending_Interval", "Modulation", "Bitrate", "Num_Sent", "Num_Reliable", "Num_Delay_Excd",
                                            "Num_Incr_Rcvd", "Num_Q_Overflow"],
                                dtype=df_dtypes)
    dataset_details_df = get_mcs_index(dataset_details_df)

    # Change sending interval categorial to numeric
    dataset_details_df["UAV_Sending_Interval_Class"] = dataset_details_df["UAV_Sending_Interval"].replace({10:0, 20:1, 66.7:2, 100:3})

    # Quantize mean and std dev of sinr
    hdist_class, hdist_bins = pd.qcut(dataset_details_df.Horizontal_Distance, q=hdist_num_bins, retbins=True, labels=False)
    height_class, height_bins = pd.qcut(dataset_details_df.Height, q=height_num_bins, retbins=True, labels=False)
    dataset_details_df["Horizontal_Distance_Class"] = hdist_class.values
    dataset_details_df["Height_Class"] = height_class.values

    # Below shows how to quantize test data
    # dataset_details_df["Horizontal_Distance_Class"] = pd.cut(dataset_details_df.Horizontal_Distance, hdist_bins, right=True, include_lowest=True, labels=False)
    # dataset_details_df["Height_Class"] = pd.cut(dataset_details_df.Height, height_bins, right=True, include_lowest=True, labels=False)


    # # Generate dataset samples
    df_train = generate_reliability_dataset(dataset_details_df)

    return df_train, hdist_bins, height_bins

# This function helps to calculate probability distribution, which goes into BBN (note, can handle up to 2 parents)
def cpt_probs(df, child, parents):
    try:
        # dependencies_arr = [pd.Categorical(df[parent],categories=df[parent].cat.categories.tolist()) for parent in parents]
        dependencies_arr = [df[parent] for parent in parents]
        # cpt = pd.crosstab(dependencies_arr, df[child], rownames=parents, colnames=[child], margins=False, normalize='index', dropna=False).sort_index().to_numpy().reshape(-1).tolist()
        cpt = pd.crosstab(dependencies_arr, df[child], rownames=parents, colnames=[child], margins=False, normalize='index', dropna=False).sort_index()
        return cpt
    except Exception as err:
        print(err)
        return None

In [2]:
DATASET_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_ParrotAR2/data_processed/ParrotAR2_{}_Reliability.csv"
SAVE_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_ParrotAR2/bn_ckpt/parrotar2_reliability_bn_{}_{}.{}"
LINKS = ["Downlink", "Uplink", "Video"]
# LINKS = ["Downlink"]
PARENTS = ["Horizontal_Distance_Class", "Height_Class", "UAV_Sending_Interval_Class", "MCS"]

for link in LINKS:
    dataset_path = DATASET_PATH.format(link)
    df_train, hdist_bins, height_bins = process_dataset(dataset_path, hdist_num_bins=121, height_num_bins=9)
    packet_state_cpt = cpt_probs(df_train, child="Packet_State", parents=PARENTS)
    np.save(SAVE_PATH.format("hdist_bins", link, "npy"), hdist_bins)
    np.save(SAVE_PATH.format("height_bins", link, "npy"), height_bins)
    packet_state_cpt.to_csv(SAVE_PATH.format("CPT", link, "csv"))

34848it [01:30, 387.14it/s]
34848it [01:22, 424.55it/s]
34848it [01:24, 412.73it/s]


Saving new hdist_bins and height_bins for new BN model and max horizontal distance of 700m.

In [4]:
import numpy as np
SAVE_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_ParrotAR2/bn_cpts/parrotar2_reliability_bn_{}_{}.{}"
LINKS = ["Downlink", "Uplink", "Video"]
HDIST_BINS = np.arange(0, 710, 10)
HDIST_BINS[-1] = HDIST_BINS[-1] + 1 # To include 700 m in the last bin
HEIGHT_BINS = np.arange(60, 330, 30)
HEIGHT_BINS[-1] = HEIGHT_BINS[-1] + 1 # To include 300 m in the last bin (left closed ended, right open ended except for final bin)
for link in LINKS:
    np.save(SAVE_PATH.format("hdist_bins", link, "npy"), HDIST_BINS)
    np.save(SAVE_PATH.format("height_bins", link, "npy"), HEIGHT_BINS)